# 02 Plotly prototype

Quick plausibility check of the star positions before building the Three.js map. If the known neighbours sit where they should, the coordinate maths is right.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

stars = pd.DataFrame(json.loads((Path("..") / "data" / "processed" / "stars.json").read_text(encoding="utf-8"))["stars"])
stars["n_planets"] = stars["planets"].str.len()
print(len(stars), "stars")

## Sanity check

Known distances from the literature: Proxima Cen about 4.2 ly, TRAPPIST-1 about 40 ly. The distance from the origin has to match.

In [ ]:
stars[stars["hostname"].isin(["Proxima Cen", "TRAPPIST-1", "Kepler-452"])][["hostname", "dist_ly", "x", "y", "z"]]

## Plot helper

Marker size uses the star radius, with a fixed default where it is missing. Square root keeps huge giants from hiding everything around them.

In [ ]:
DEFAULT_RADIUS = 1.0  # solar radii, only used for display when st_rad is missing

def star_map(df, title):
    radius = df["st_rad"].fillna(DEFAULT_RADIUS)
    size = 2 + 3 * np.sqrt(radius.clip(upper=25))

    hover = (
        df["hostname"] + "<br>" + df["dist_ly"].round(1).astype(str) + " ly<br>"
        + df["n_planets"].astype(str) + " planet(s)"
        + np.where(df["teff_missing"], "<br>temperature unknown", "")
    )

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=df["x"], y=df["y"], z=df["z"], mode="markers",
        marker=dict(size=size, color=df["color"], opacity=0.85, line=dict(width=0)),
        text=hover, hoverinfo="text", name="stars",
    ))
    # The Sun is the origin of the map, so it gets its own marker.
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode="markers", name="Sun",
        marker=dict(size=6, color="white", line=dict(color="gold", width=3)),
        hoverinfo="name",
    ))
    fig.update_layout(
        title=title, template="plotly_dark", height=650, showlegend=False,
        scene=dict(aspectmode="data", xaxis_title="x (ly)", yaxis_title="y (ly)", zaxis_title="z (ly)"),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig

## Neighbourhood, up to 500 light years

Most of the discoveries sit here, so this is the readable view.

In [ ]:
near = stars[stars["dist_ly"] <= 500]
print(len(near), "stars")
star_map(near, "Exoplanet host stars within 500 ly")

## All stars

The far field shows the problem named in the project document: a few thousand light years of range squeeze the neighbourhood into the centre. A log scale option is planned for the web map.

In [ ]:
star_map(stars, "All exoplanet host stars")